In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import lightning as L
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split

/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = "./iris.txt"
df = pd.read_table(url, sep=",", header=None)

In [3]:
df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
df.columns = ["sepal_length",
              "sepal_width",
              "petal_length",
              "petal_width",
              "class"]
df.head

<bound method NDFrame.head of      sepal_length  sepal_width  petal_length  petal_width           class
0             5.1          3.5           1.4          0.2     Iris-setosa
1             4.9          3.0           1.4          0.2     Iris-setosa
2             4.7          3.2           1.3          0.2     Iris-setosa
3             4.6          3.1           1.5          0.2     Iris-setosa
4             5.0          3.6           1.4          0.2     Iris-setosa
..            ...          ...           ...          ...             ...
145           6.7          3.0           5.2          2.3  Iris-virginica
146           6.3          2.5           5.0          1.9  Iris-virginica
147           6.5          3.0           5.2          2.0  Iris-virginica
148           6.2          3.4           5.4          2.3  Iris-virginica
149           5.9          3.0           5.1          1.8  Iris-virginica

[150 rows x 5 columns]>

In [5]:
df.shape

(150, 5)

In [6]:
df['class'].nunique()

3

In [7]:
df['class'].unique()

array(['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'], dtype=object)

In [8]:
for class_name in df['class'].unique():
    print(class_name, ": ", sum(df['class'] == class_name), sep="")

Iris-setosa: 50
Iris-versicolor: 50
Iris-virginica: 50


In [9]:
df[['petal_width', 'sepal_width']].head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [10]:
input_values = df[['petal_width', 'sepal_width']]
input_values.head()

,petal_width,sepal_width
0,0.2,3.5
1,0.2,3.0
2,0.2,3.2
3,0.2,3.1
4,0.2,3.6


In [11]:
label_values = df['class']
label_values.head()

0    Iris-setosa
1    Iris-setosa
2    Iris-setosa
3    Iris-setosa
4    Iris-setosa
Name: class, dtype: object

In [12]:
class_as_numbers = label_values.factorize()[0]
class_as_numbers

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [13]:
input_train, input_test, label_train, label_test = train_test_split(input_values,
                                                                    class_as_numbers,
                                                                    test_size=0.25,
                                                                    stratify=class_as_numbers,
                                                                    random_state=42)

In [14]:
input_train.shape

(112, 2)

In [15]:
label_train.shape

(112,)

In [16]:
input_test.shape

(38, 2)

In [17]:
label_test.shape

(38,)

In [18]:
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

In [19]:
one_hot_label_train[:10]

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]])

In [20]:
max_vlaues_in_input_train = input_train.max()
max_vlaues_in_input_train

petal_width    2.5
sepal_width    4.4
dtype: float64

In [21]:
min_values_in_input_train = input_train.min()
min_values_in_input_train

petal_width    0.1
sepal_width    2.0
dtype: float64

In [22]:
input_train =   (input_train - min_values_in_input_train) / (max_vlaues_in_input_train - min_values_in_input_train)
input_train.head()

,petal_width,sepal_width
130,0.750000,0.333333
122,0.791667,0.333333
81,0.375000,0.166667
71,0.500000,0.333333
89,0.500000,0.208333


In [23]:
input_test = (input_test - min_values_in_input_train) / (max_vlaues_in_input_train - min_values_in_input_train)
input_test.head()

,petal_width,sepal_width
42,0.041667,0.500000
56,0.625000,0.541667
99,0.500000,0.333333
53,0.500000,0.125000
38,0.041667,0.416667


In [24]:
input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_train_tensors[:5]

tensor([[0.7500, 0.3333],
        [0.7917, 0.3333],
        [0.3750, 0.1667],
        [0.5000, 0.3333],
        [0.5000, 0.2083]])

In [25]:
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)
input_test_tensors[:5]

tensor([[0.0417, 0.5000],
        [0.6250, 0.5417],
        [0.5000, 0.3333],
        [0.5000, 0.1250],
        [0.0417, 0.4167]])

In [26]:
train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

In [27]:
class MultipleInsoOuts(L.LightningModule):
    
    def __init__(self):
        super().__init__()
        L.seed_everything(seed=42)
        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)
        self.loss = nn.MSELoss(reduction='sum')
        
    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))
        return (output_values)
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=1e-2)
    
    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)
        return loss

In [28]:
model = MultipleInsoOuts()

for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

Seed set to 42


input_to_hidden.weight tensor([[ 0.5400,  0.5900],
        [-0.1700,  0.6500]])
input_to_hidden.bias tensor([-0.1500,  0.1400])
hidden_to_output.weight tensor([[-0.3400,  0.4200],
        [ 0.6200, -0.5200],
        [ 0.6100,  0.1300]])
hidden_to_output.bias tensor([0.5200, 0.1000, 0.3400])


In [29]:
model = MultipleInsoOuts()

Seed set to 42


In [30]:
trainer = L.Trainer(max_epochs=10)
trainer.fit(model, train_dataloaders=train_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoin

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_to_hidden  │ Linear  │      6 │ train │     0 │
│ 1 │ hidden_to_output │ Linear  │      9 │ train │     0 │
│ 2 │ loss             │ MSELoss │      0 │ train │     0 │
└───┴──────────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 15                                                                                               
Non-trainable params: 0                                                                                            
Total params: 15                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

`Trainer.fit` stopped: `max_epochs=10` reached.


In [31]:
predictions = model(input_test_tensors)

In [32]:
predictions[0:4,]

tensor([[ 0.9237,  0.0700, -0.0135],
        [ 0.0339,  0.5115,  0.3848],
        [ 0.0787,  0.6557,  0.1132],
        [ 0.0780,  0.6535,  0.1173]], grad_fn=<SliceBackward0>)

In [33]:
predicted_labels = torch.argmax(predictions, dim=1)
predicted_labels[0:4]

tensor([0, 1, 1, 1])

In [34]:
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)

tensor(0.9474)

In [35]:
path_to_checkpoint = trainer.checkpoint_callback.best_model_path

In [36]:
trainer = L.Trainer(max_epochs=100)
trainer.fit(model, train_dataloaders=train_dataloader, ckpt_path = path_to_checkpoint)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at /Users/chengtechang/Desktop/vs/DL_ch/lightning_logs/version_0/checkpoints/epoch=9-step=1120.ckpt
/Users/chengtechang/Desktop/vs/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/Users/chengtechang/Desktop/vs/DL_ch/lightning_logs/version_0/checkpoints' to '/Users/chengtechang/Desktop/vs/DL_ch/lightning_logs/version_1/checkpoints', therefore `best_model_score`, `kth_best_

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_to_hidden  │ Linear  │      6 │ train │     0 │
│ 1 │ hidden_to_output │ Linear  │      9 │ train │     0 │
│ 2 │ loss             │ MSELoss │      0 │ train │     0 │
└───┴──────────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 15                                                                                               
Non-trainable params: 0                                                                                            
Total params: 15                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 3                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restored all states from the checkpoint at /Users/chengtechang/Desktop/vs/DL_ch/lightning_logs/version_0/checkpoints/epoch=9-step=1120.ckpt
`Trainer.fit` stopped: `max_epochs=100` reached.


In [37]:
predictions = model(input_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(label_test), predicted_labels)) / len(predicted_labels)


tensor(0.9474)

In [38]:
normalized_values = ([0.2,3.0] - min_values_in_input_train) / (max_vlaues_in_input_train - min_values_in_input_train)
normalized_values

petal_width    0.041667
sepal_width    0.416667
dtype: float64

In [39]:
model(torch.tensor(normalized_values).type(torch.float32))

/var/folders/sk/j94l_d853vx6kdmz54x33ytw0000gn/T/ipykernel_65460/3219751161.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model(torch.tensor(normalized_values).type(torch.float32))


tensor([ 1.0095, -0.0195, -0.0099], grad_fn=<ViewBackward0>)